## Creating & Running a Team: RoundRobinGroupChat

In [2]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")
load_dotenv()

api_key = os.environ.get("AICREDITS_API_KEY")
base_url = os.environ.get("BASE_URL")

if not api_key:
    raise ValueError("Please set the OpenAI_API_KEY")

In [3]:
model_client = OpenAIChatCompletionClient(model="gpt-5-nano", api_key=api_key, base_url=base_url)

dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words."
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the `Complex_DSA_Solver` and make sure it is optimized. Give under 10 words."
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required. Give under 10 words"
)

### max_turns=3

In [9]:
team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    max_turns=3
)

async def run_team():
    task = TextMessage(content='write a simple code in python to add 2 numbers', source='user')
    result = await team.run(task=task)
    print(result)

await run_team()

messages=[TextMessage(id='0b336957-a4f3-4f4e-aa80-7bcfb43c417f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 9, 26, 12, 1, 48, 47077, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'), TextMessage(id='f6b84e84-072f-4606-9777-08d236cf1593', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=40, completion_tokens=754), metadata={}, created_at=datetime.datetime(2026, 9, 26, 12, 1, 54, 389398, tzinfo=datetime.timezone.utc), content='```python\n# Simple Python program to add two numbers\na = float(input("Enter first number: "))\nb = float(input("Enter second number: "))\nprint("Sum =", a + b)\n```', type='TextMessage'), TextMessage(id='98cf8175-03dc-40bf-aeb1-2e29c80a281b', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=100, completion_tokens=1109), metadata={}, created_at=datetime.datetime(2026, 9, 26, 12, 2, 3, 839864, tzinfo=datetime.timezone.utc), content='

### max_turn=1

In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    max_turns=1   
)


async def run_team():
    task = TextMessage(content='write a simple code in python to add 2 numbers',source='user')
    result = await team.run(task=task)
    print(result)

await run_team()

messages=[TextMessage(id='e79ce0b9-acaa-4fa0-99ab-7bfcc83ec5a3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 9, 26, 12, 20, 18, 147688, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'), TextMessage(id='c1da99d9-0b72-4fd1-a2f9-ae46b7de3a39', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=489), metadata={}, created_at=datetime.datetime(2026, 9, 26, 12, 20, 23, 752648, tzinfo=datetime.timezone.utc), content='```python\na = float(input("Enter first number: "))\nb = float(input("Enter second number: "))\nprint("Sum =", a + b)\n```', type='TextMessage')] stop_reason='Maximum number of turns 1 reached.'


## TextMentionTermination
Stops when specific text or string is mentioned in a message (e.g., “TERMINATE”).

In [ ]:
code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the `Complex_DSA_Solver` and make sure it is optimized. Give under 10 words. If you feel that the code is fine, please say 'TERMINATE'"
)

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination

my_termination = TextMentionTermination(text='TERMINATE') 

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)


async def run_team():
    task = TextMessage(content='write a code to find median of two sorted arrays',source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")

await run_team()

user : write a code to find median of two sorted arrays
Complex_DSA_Solver : ```python
def median_of_two_sorted_arrays(a, b):
    m, n = len(a), len(b)
    if m == 0 and n == 0:
        raise ValueError("Both arrays empty")
    i = j = 0
    k = (m + n) // 2
    prev = curr = 0
    for _ in range(k + 1):
        prev = curr
        if i < m and (j == n or a[i] <= b[j]):
            curr = a[i]; i += 1
        else:
            curr = b[j]; j += 1
    return curr if (m + n) % 2 == 1 else (prev + curr) / 2.0
```
CODE_REVIEWER : Use O(log(m+n)) binary search; current is O(m+n).
CODE_EDITOR : ```python
# Median of two sorted arrays, O(log(min(m,n)))
def median_of_two_sorted_arrays(a, b):
    if len(a) > len(b):
        a, b = b, a  # ensure a is the smaller array
    m, n = len(a), len(b)
    if m == 0:
        if n == 0:
            raise ValueError("Both arrays empty")
        if n % 2 == 1:
            return float(b[n // 2])
        else:
            return (b[n // 2 - 1] + b[n // 2]) 

## MaxMessageTermination
Stops after a specified number of messages have been produced, including both agent and task messages.

In [ ]:
code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the `Complex_DSA_Solver` and make sure it is optimized. Give under 10 words."
)

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import MaxMessageTermination

my_termination = MaxMessageTermination(max_messages=2)

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    # max_turns=6
)

async def run_team():
    task = TextMessage(content='write a code to add 2 numbers. TERMINATE',source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")

await run_team()

user : write a code to add 2 numbers. TERMINATE
Complex_DSA_Solver : ```python
# Add two numbers
a = float(input("Enter first number: "))
b = float(input("Enter second number: "))
print("Sum =", a + b)
```


In [15]:
# team defined above  
from autogen_agentchat.base import TaskResult

async for message in team.run_stream(task="Write a simple Hello world code?"):
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)
    else:
        print(message.source, message)

user id='1a9de72d-3dcd-43ce-b223-ba7d75a332ae' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 9, 26, 13, 16, 53, 337300, tzinfo=datetime.timezone.utc) content='Write a simple Hello world code?' type='TextMessage'
CODE_REVIEWER id='5c8286a0-3b4d-425a-9632-4c954123a795' source='CODE_REVIEWER' models_usage=RequestUsage(prompt_tokens=108, completion_tokens=1043) metadata={} created_at=datetime.datetime(2026, 9, 26, 13, 17, 1, 74722, tzinfo=datetime.timezone.utc) content='```python\nprint("Hello, World!")\n```' type='TextMessage'
Stop Reason: Maximum number of messages 2 reached, current message count: 2


In [ ]:
state = await team.save_state()
state

## TextMentionTermination | MaxMessageTermination

In [27]:
dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words."
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the `Complex_DSA_Solver` and make sure it is optimized. Give under 10 words. If you feel that the code is fine, please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required. Give under 10 words"
)

In [29]:
my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=9)

team_2 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
)

async def run_team():
    task = TextMessage(content='write a code to add 2 numbers',source='user')
    result = await team_2.run(task=task)
    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")

await run_team()

user : write a code to add 2 numbers
Complex_DSA_Solver : def add(a, b):
    return a + b

if __name__ == "__main__":
    a = float(input("Enter first number: "))
    b = float(input("Enter second number: "))
    print("Sum:", add(a, b))
CODE_REVIEWER : TERMINATE


## Just For Understanding

In [ ]:
from autogen_agentchat.agents import AssistantAgent

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=api_key, base_url=base_url)

add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)

my_increment_team = RoundRobinGroupChat(participants=[add_1_agent_first,add_1_agent_second], max_turns=2)

In [41]:
from autogen_agentchat.ui import Console

await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2


TaskResult(messages=[TextMessage(id='660b7df0-2326-40f7-a9fc-0f1f53816468', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 5, 465602, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='95a5b6b2-c2b6-4db8-b695-adea3c39e021', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 6, 509232, tzinfo=datetime.timezone.utc), content='2', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [42]:
# await my_increment_team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
3
---------- TextMessage (add_1_agent_second) ----------
4


TaskResult(messages=[TextMessage(id='7472c7a2-be5f-4600-a233-1bb2500702d2', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 14, 138463, tzinfo=datetime.timezone.utc), content='3', type='TextMessage'), TextMessage(id='3bfc3fc2-26f1-4dbe-b4a2-ae0548814c43', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=49, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 14, 993583, tzinfo=datetime.timezone.utc), content='4', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [43]:
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
5
---------- TextMessage (add_1_agent_second) ----------
6


TaskResult(messages=[TextMessage(id='c1b70192-ec63-45dc-a089-26f6a8165ff1', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=54, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 22, 420044, tzinfo=datetime.timezone.utc), content='5', type='TextMessage'), TextMessage(id='4402247b-d005-4a55-837f-9450bdeaf48c', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 23, 455095, tzinfo=datetime.timezone.utc), content='6', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

## Resetting a Team

In [44]:
await my_increment_team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2


TaskResult(messages=[TextMessage(id='c9f80d93-c79c-400c-abf4-12b136e9b623', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 52, 129261, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='f0d0e4e5-b7eb-45cd-949a-602dbc1a6cb8', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 9, 26, 16, 12, 53, 289993, tzinfo=datetime.timezone.utc), content='2', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

## Stopping a Team
Stop the team from outside by using the ExternalTermination.

In [5]:
code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the `Complex_DSA_Solver` and make sure it is optimized. Give under 10 words. If you feel that the code is fine, please say 'TERMINATE'"
)

In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination, ExternalTermination, TextMentionTermination
from autogen_agentchat.ui import Console

external_termination = ExternalTermination()
my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=9)

team_3 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=external_termination | my_termination,  # Use the bitwise OR operator to combine conditions
)

run = asyncio.create_task(Console(team_3.run_stream(task="write a code to find median of two sorted arrays")))

# Wait for some time.
await asyncio.sleep(0.1)

# Stop the team.
external_termination.set()

# Wait for the team to finish.
await run


# Output: stop_reason='External termination requested'

---------- TextMessage (user) ----------
write a code to find median of two sorted arrays
---------- TextMessage (Complex_DSA_Solver) ----------
def find_median_sorted_arrays(a, b):
    n, m = len(a), len(b)
    total = n + m
    if total == 0: return None
    i = j = 0
    mid = total // 2
    prev = cur = 0
    while i < n or j < m:
        if j >= m or (i < n and a[i] <= b[j]):
            prev, cur = cur, a[i]
            i += 1
        else:
            prev, cur = cur, b[j]
            j += 1
        if total % 2 == 1 and i + j - 1 == mid:
            return cur
        if total % 2 == 0 and i + j - 1 == mid:
            return (cur + prev) / 2.0


TaskResult(messages=[TextMessage(id='86a56395-a93d-49ae-b616-13ab88c523f6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 9, 27, 12, 27, 19, 254663, tzinfo=datetime.timezone.utc), content='write a code to find median of two sorted arrays', type='TextMessage'), TextMessage(id='24873125-31e5-4dda-ba1a-328bf3f2cfa4', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=4540), metadata={}, created_at=datetime.datetime(2026, 9, 27, 12, 27, 43, 700850, tzinfo=datetime.timezone.utc), content='def find_median_sorted_arrays(a, b):\n    n, m = len(a), len(b)\n    total = n + m\n    if total == 0: return None\n    i = j = 0\n    mid = total // 2\n    prev = cur = 0\n    while i < n or j < m:\n        if j >= m or (i < n and a[i] <= b[j]):\n            prev, cur = cur, a[i]\n            i += 1\n        else:\n            prev, cur = cur, b[j]\n            j += 1\n        if total % 2 == 1 and i + j - 1 == mid:\n            r

## Resuming a Team 
- Teams are stateful and maintains the conversation history and context after each run, unless you reset the team.

Resume a team to continue from where it left off by calling the `run()` or `run_stream()` method again without a new task

In [ ]:
await Console(team_3.run_stream())

## Output: stop_reason="Text 'TERMINATE' mentioned"

---------- TextMessage (CODE_REVIEWER) ----------
TERMINATE


TaskResult(messages=[TextMessage(id='c2e1cd32-ccd5-44de-a461-c771e9d842b4', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=253, completion_tokens=588), metadata={}, created_at=datetime.datetime(2026, 9, 27, 12, 32, 32, 400118, tzinfo=datetime.timezone.utc), content='TERMINATE', type='TextMessage')], stop_reason="Text 'TERMINATE' mentioned")

## Aborting a Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [10]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_core import CancellationToken

cancellation_token = CancellationToken()
my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=9)

team_3 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination
)

run = asyncio.create_task(team_3.run(task="write a code to find median of two sorted arrays", cancellation_token=cancellation_token,))

# Cancel the run.
cancellation_token.cancel()

try:
    result = await run  # This will raise a CancelledError.
except asyncio.CancelledError:
    print("Task was cancelled.")

Task was cancelled.


## Single-Agent Team

In [11]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-5-nano", api_key=api_key, base_url=base_url,  parallel_tool_calls=False)

def increment_number(number: int) -> int:
    """Increment a number by 1."""
    return number + 1

looped_assistant = AssistantAgent(
    "looped_assistant",
    model_client=model_client,
    tools=[increment_number],
    system_message="You are a helpful AI assistant, use the tool to increment the number.",
)

termination_condition = TextMessageTermination("looped_assistant")

team_4 = RoundRobinGroupChat(
    participants=[looped_assistant],
    termination_condition=termination_condition,
)

run = asyncio.create_task(Console(team_4.run_stream(task="Increment the number from 5 to 10")))

await run

---------- TextMessage (user) ----------
Increment the number from 5 to 10
---------- ToolCallRequestEvent (looped_assistant) ----------
[FunctionCall(id='call_F54fQN6fC0dqcMww032uNf3V', arguments='{"number":5}', name='increment_number')]
---------- ToolCallExecutionEvent (looped_assistant) ----------
[FunctionExecutionResult(content='6', name='increment_number', call_id='call_F54fQN6fC0dqcMww032uNf3V', is_error=False)]
---------- ToolCallSummaryMessage (looped_assistant) ----------
6
---------- ToolCallRequestEvent (looped_assistant) ----------
[FunctionCall(id='call_n03epCunuFr1bC0FJb8ysNmb', arguments='{"number":6}', name='increment_number')]
---------- ToolCallExecutionEvent (looped_assistant) ----------
[FunctionExecutionResult(content='7', name='increment_number', call_id='call_n03epCunuFr1bC0FJb8ysNmb', is_error=False)]
---------- ToolCallSummaryMessage (looped_assistant) ----------
7
---------- ToolCallRequestEvent (looped_assistant) ----------
[FunctionCall(id='call_WFJ5iuPF0r

TaskResult(messages=[TextMessage(id='f02f32ad-d094-40b5-a36a-b439f5ad5237', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 9, 27, 13, 34, 40, 960615, tzinfo=datetime.timezone.utc), content='Increment the number from 5 to 10', type='TextMessage'), ToolCallRequestEvent(id='b2dab65f-83f9-446a-b703-60ccb8634235', source='looped_assistant', models_usage=RequestUsage(prompt_tokens=159, completion_tokens=599), metadata={}, created_at=datetime.datetime(2026, 9, 27, 13, 34, 45, 476947, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_F54fQN6fC0dqcMww032uNf3V', arguments='{"number":5}', name='increment_number')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='5340106a-6e94-4c5c-8264-8fe148b69599', source='looped_assistant', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 9, 27, 13, 34, 45, 486756, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='6', name='increment_number', call_id='call_F54fQN6f